# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManjusreeValluri/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
%pip -q install duckdb

In [17]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected to Hugging Face")

DuckDB connected to Hugging Face


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row represents the daily performance of one content item for one client on one report date.

**Time window:** I will use March 2026 (`2026-03`) as the development window. I will treat June 2026 as the final/sealed month and will not use it to develop the label or features.


In [18]:
# Verify that each report_date + client + content combination is unique

grain_check = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_per_grain
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_result = con.execute(grain_check).fetchdf()
grain_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,rows_per_grain


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions`

**Label / proxy:** A future-performance proxy will be constructed from later observations; it will not be used as an input feature.

**Context:** `report_date`, `client_hash_id`, `content_hash_id`, `gsc_data_available`, `ga4_data_available`

**Excluded:** Future observations and any outcome-derived field are excluded from the features because they would leak information that would not be available at the decision moment.


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify March 2026 row count and date span

march_check = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

march_result = con.execute(march_check).fetchdf()
march_result


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [21]:
# Verify availability using IS TRUE

availability_check = """
SELECT
    COUNT(*) AS march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4_data
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

availability_result = con.execute(availability_check).fetchdf()
availability_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,rows_with_ga4_data
0,9841378,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

This slice cannot tell us whether a content change actually caused a later performance change because the warehouse contains observational performance data rather than a controlled experiment. Client history also varies, and some rows have limited or unavailable GSC/GA4 data, so results should be treated as decision-support rather than proof of causation.


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [23]:
schema_check = """
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
"""

schema = con.execute(schema_check).fetchdf()
schema[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [24]:
# Build the five-feature frame for March 2026
# Read only the March partition

feature_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10000
"""

features = con.execute(feature_query).fetchdf()

print("Rows loaded:", len(features))
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 10000


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>


### Five features and why they are available at the decision moment

* **`gsc_impressions`** — knowable at the decision moment because it records search impressions already observed for the content.
* **`gsc_clicks`** — knowable at the decision moment because it records search clicks already observed for the content.
* **`gsc_avg_position`** — knowable at the decision moment because it records the observed Google Search Console position for the content.
* **`ga4_sessions`** — knowable at the decision moment because it records sessions already observed by that reporting date.
* **`ga4_engaged_sessions`** — knowable at the decision moment because it records engaged sessions already observed by that reporting date.


In [25]:
# Create a small March dataset for the leakage demonstration

leak_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
LIMIT 10000
"""

leak_df = con.execute(leak_query).fetchdf()

print("Rows:", len(leak_df))
leak_df.head()

Rows: 10000


,report_date,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,0,20,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,0,1,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,1,125,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,0,7,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0,11,2.272727,<NA>,<NA>


In [26]:
# Create a future-click decline label

label_query = """
WITH current_rows AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks AS current_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
      AND report_date <= DATE '2026-03-24'
    LIMIT 10000
),

future_rows AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks AS future_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
)

SELECT
    c.report_date,
    c.client_hash_id,
    c.content_hash_id,
    c.current_clicks,
    f.future_clicks,
    CASE
        WHEN f.future_clicks < c.current_clicks THEN 1
        ELSE 0
    END AS decline_label
FROM current_rows c
LEFT JOIN future_rows f
    ON c.client_hash_id = f.client_hash_id
    AND c.content_hash_id = f.content_hash_id
    AND f.report_date = c.report_date + INTERVAL 7 DAY
WHERE f.future_clicks IS NOT NULL
"""

label_df = con.execute(label_query).fetchdf()

print("Rows with a future outcome:", len(label_df))
print("Label distribution:")
print(label_df["decline_label"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows with a future outcome: 9281
Label distribution:
decline_label
0    8520
1     761
Name: count, dtype: int64


In [27]:
# Create the label dataset with the five honest features
# plus future_clicks for the deliberate leakage experiment.

label_query = """
WITH current_rows AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        ga4_engaged_sessions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
      AND report_date <= DATE '2026-03-24'
    LIMIT 10000
),

future_rows AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_clicks AS future_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
)

SELECT
    c.report_date,
    c.client_hash_id,
    c.content_hash_id,
    c.gsc_impressions,
    c.gsc_clicks,
    c.gsc_avg_position,
    c.ga4_sessions,
    c.ga4_engaged_sessions,
    f.future_clicks,
    CASE
        WHEN f.future_clicks < c.gsc_clicks THEN 1
        ELSE 0
    END AS decline_label
FROM current_rows c
LEFT JOIN future_rows f
    ON c.client_hash_id = f.client_hash_id
    AND c.content_hash_id = f.content_hash_id
    AND f.report_date = c.report_date + INTERVAL 7 DAY
WHERE f.future_clicks IS NOT NULL
"""

label_df = con.execute(label_query).fetchdf()

print("Rows with a future outcome:", len(label_df))
print("Columns:", list(label_df.columns))
print("Label distribution:")
print(label_df["decline_label"].value_counts())

Rows with a future outcome: 9281
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions', 'future_clicks', 'decline_label']
Label distribution:
decline_label
0    8520
1     761
Name: count, dtype: int64


### Leakage lesson

I deliberately included `future_clicks`, which is derived from seven days after the decision moment. This is a leakage trap because that information would not be available when making the real decision. The leaky model's score is therefore not an honest estimate of performance.

After removing `future_clicks`, I kept the five features that are available at the decision moment. The honest model score is the number I will retain for this exercise.
